In [ ]:
library(tidyverse)

In [ ]:
library("dbplyr")
library("RPostgres")
library("toml")

In [ ]:
cfg <- toml::read_toml("../config.toml")

In [ ]:
u <- httr2::url_parse(cfg$database_url)

In [ ]:
con <- dbConnect(
  RPostgres::Postgres(),
  host     = u$hostname,
  port     = if (is.null(u$port)) 5432L else as.integer(u$port),
  dbname   = sub("^/", "", u$path),
  user     = u$username,
  password = u$password,
  sslmode  = if (isTRUE(cfg$pg_tls)) "require" else "prefer")

In [ ]:
readings <- tbl(con, "office_fishtank_readings")

In [ ]:
readings

In [ ]:
readings %>%
  filter(time >= local(Sys.time() - 2 * 3600)) %>%
  filter(sensor %in% c("water_temperature", "tds", "ph")) %>%
  arrange(time) %>%
  collect()

In [ ]:
readings %>%
  filter(time >= local(Sys.time() - 72 * 3600)) %>%
  filter(sensor %in% c("water_temperature")) %>%
  arrange(time) %>%
  collect() %>%
  ggplot(aes(x = time, y = value)) +
    geom_line()

In [ ]:
readings %>%
  filter(time >= local(Sys.time() - 72 * 3600)) %>%
  filter(sensor %in% c("tds")) %>%
  arrange(time) %>%
  collect() %>%
  ggplot(aes(x = time, y = value)) +
    geom_line()

In [ ]:
readings %>%
  filter(time >= local(Sys.time() - 72 * 3600)) %>%
  filter(sensor %in% c("ph")) %>%
  arrange(time) %>%
  collect() %>%
  ggplot(aes(x = time, y = value)) +
    geom_line()

# Trigger a reading

In [ ]:
m <- cfg$mqtt

In [ ]:
args <- c(
  "-h", m$host,
  "-p", m$port,
  "-i", paste0("r_trigger_", Sys.getpid()),
  "-t", "fishtank/command/measure",
  "-q", "1",
  "-m", "1",
  "-u", m$user,
  "-P", m$password)

In [ ]:
system2("/opt/homebrew/bin/mosquitto_pub",
    args,
    stdout = "", stderr = "")

Sys.time()